# Spezialfälle

Dieses Notebook demonstriert Randsituationen, die im normalen Wahlablauf selten auftreten, aber spezifisches Verhalten der Simulation auslösen.

**Grundprinzip:** 1 Wahlkreis, 100 Stimmen → Stimmanzahl = Prozentzahl. Gewinnerschwelle: **52 %** (Standard: `ballot_majority_margin = 2 %`).

In [ ]:
import pandas as pd

from ipres import (
    Election, ElectionConfig,
    Ballot, DrawOfLots, DrawLotsStrategy,
    ElectionEvaluator,
)
from ipres.constituencies_config import ConstituenciesConfig

# 1 Wahlkreis, 100 Stimmen → Stimmanzahl = Prozentzahl
cc = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': ['WK'],
    'constituency_size': [100],
    'turnout_percent':   [100.0],
}))

---
## Fall 1: Nur eine Partei — `ValueError`

`Ballot.run()` erwartet mindestens zwei Wahlteilnehmer. Startet man eine Wahl mit nur einer Partei, wird ein `ValueError` ausgelöst.

In [ ]:
config_1 = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A'],
)

try:
    election_1 = Election(electionConfig=config_1)
    election_1.start(votes={'A': 100})
except ValueError as e:
    print(f"ValueError: {e}")

---
## Fall 2a: Zwei Parteien — Sieg im ersten Wahlgang

| Partei | Stimmen | Anteil |
|--------|:-------:|:------:|
| A      | 53      | 53 %   |
| B      | 47      | 47 %   |

A übertrifft die Schwelle von **52 %** direkt → Sieg in Runde 1.

In [ ]:
config_2a = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
)

election_2a = Election(electionConfig=config_2a)
r1_2a = election_2a.start(votes={'A': 53, 'B': 47})

print(f"Schwelle: {config_2a.getBallotMajorityPercent():.0f} %")
print(r1_2a.getContestantsVotesAfterPossibleCoalitions().to_string())
print(f"\nGewinner: {r1_2a.getWinner().name}")
print(f"Runden gesamt: {election_2a.getNumberOfIterations()}")

---
## Fall 2b: Zwei Parteien — Sieg im zweiten Wahlgang

| Wahlgang | A  | B  | Ergebnis                     |
|:--------:|:--:|:--:|:----------------------------:|
| 1        | 51 | 49 | kein Gewinner (51 % < 52 %)  |
| 2        | 55 | 45 | **A gewinnt (55 % ≥ 52 %)**  |

Im ersten Wahlgang verfehlt A die Schwelle knapp. Im zweiten Durchgang reicht es.

In [ ]:
config_2b = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
)

election_2b = Election(electionConfig=config_2b)

r1_2b = election_2b.start(votes={'A': 51, 'B': 49})
print(f"Runde 1: Gewinner={r1_2b.getWinner()}  → zweiter Wahlgang nötig")

r2_2b = election_2b.runNextIteration(
    r1_2b.getNextRoundInput().with_votes({'A': 55, 'B': 45})
)
print(f"Runde 2: Gewinner={r2_2b.getWinner().name}")
print(f"Runden gesamt: {election_2b.getNumberOfIterations()}")

---
## Fall 2c: Zwei Parteien — Losentscheid

Zwei Parteien, bei denen in **zwei aufeinanderfolgenden Wahlgängen** kein Gewinner ermittelt werden kann, lösen im dritten Durchgang automatisch einen Losentscheid aus — da keine Reduktion auf weniger als zwei Parteien möglich ist.

| Wahlgang | A  | B  | Ergebnis                    |
|:--------:|:--:|:--:|:---------------------------:|
| 1        | 51 | 49 | kein Gewinner (51 % < 52 %) |
| 2        | 51 | 49 | kein Gewinner               |
| 3        | —  | —  | **Losentscheid**            |

### Fall 2cI: `DrawLotsStrategy.RANDOM`

Der Gewinner wird per gleichverteiltem Zufallszug bestimmt. Mit festem `seed` ist das Ergebnis reproduzierbar.

In [ ]:
config_2cI = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
    draw_lots_strategy=DrawLotsStrategy.RANDOM,
    seed=42,
)

election_2cI = Election(electionConfig=config_2cI)
stimmen = {'A': 51, 'B': 49}

r1_I = election_2cI.start(votes=stimmen)
r2_I = election_2cI.runNextIteration(r1_I.getNextRoundInput().with_votes(stimmen))

print(f"Runde 2 → Losentscheid im nächsten Durchgang: {r2_I.needsDecisionByLotInNextRound()}")

r3_I = election_2cI.runNextIteration(r2_I.getNextRoundInput())

print(f"Runde 3 ({type(r3_I).__name__}): Gewinner={r3_I.getWinner().name}")
print(f"Durch Los entschieden: {r3_I.wasDecidedByLot()}")
print(f"Runden gesamt: {election_2cI.getNumberOfIterations()}")

### Fall 2cII: `DrawLotsStrategy.MARGINAL_LEAD`

Der marginale Stimmunterschied gilt als zufällig entstanden. Die Partei mit dem höheren Stimmenanteil im vorangegangenen Wahlgang gewinnt — hier deterministisch A, da 51 > 49.

In [ ]:
config_2cII = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B'],
    draw_lots_strategy=DrawLotsStrategy.MARGINAL_LEAD,
)

election_2cII = Election(electionConfig=config_2cII)
stimmen = {'A': 51, 'B': 49}

r1_II = election_2cII.start(votes=stimmen)
r2_II = election_2cII.runNextIteration(r1_II.getNextRoundInput().with_votes(stimmen))
r3_II = election_2cII.runNextIteration(r2_II.getNextRoundInput())

print(f"Runde 3 ({type(r3_II).__name__}): Gewinner={r3_II.getWinner().name}")
print(f"  → A gewinnt deterministisch (51 > 49), kein Zufallszug nötig")
print(f"Durch Los entschieden: {r3_II.wasDecidedByLot()}")

---
## Fall 3: Drei Parteien — Eliminierung gefolgt von Losentscheid

Dieser Fall kombiniert zwei Mechanismen: Zuerst scheidet eine Partei durch die ⅔-Regel aus, danach führen zwei verbleibende Parteien zu einem Losentscheid.

| Wahlgang | A  | B  | C  | Ergebnis                                   |
|:--------:|:--:|:--:|:--:|:------------------------------------------:|
| 1        | 40 | 38 | 22 | kein Gewinner; C ausgeschieden (⅔-Regel)   |
| 2        | 51 | 49 | —  | kein Gewinner                              |
| 3        | 51 | 49 | —  | kein Gewinner                              |
| 4        | —  | —  | —  | **Losentscheid**                           |

**⅔-Regel in Runde 1:** A (40 %) + B (38 %) = 78 % ≥ 66,7 % → C (22 %) scheidet aus.  
Ein Losentscheid wird erst nach **zwei** aufeinanderfolgenden Runden mit denselben zwei Parteien ohne Gewinner ausgelöst.

In [ ]:
config_3 = ElectionConfig(
    constituencies_config=cc,
    participating_parties=['A', 'B', 'C'],
    draw_lots_strategy=DrawLotsStrategy.RANDOM,
    seed=7,
)

election_3 = Election(electionConfig=config_3)
ab_stimmen = {'A': 51, 'B': 49}

# Runde 1: C scheidet aus
r1_3 = election_3.start(votes={'A': 40, 'B': 38, 'C': 22})
print(f"Runde 1: Noch im Rennen nach Eliminierung: {list(r1_3.getNextRoundInput().contestants.keys())}")

# Runde 2 und 3: A und B ohne Gewinner
r2_3 = election_3.runNextIteration(r1_3.getNextRoundInput().with_votes(ab_stimmen))
r3_3 = election_3.runNextIteration(r2_3.getNextRoundInput().with_votes(ab_stimmen))

print(f"Runde 2: Gewinner={r2_3.getWinner()}")
print(f"Runde 3: Gewinner={r3_3.getWinner()}  → Losentscheid im nächsten Durchgang: {r3_3.needsDecisionByLotInNextRound()}")

# Runde 4: Losentscheid
r4_3 = election_3.runNextIteration(r3_3.getNextRoundInput())

print(f"\nRunde 4 ({type(r4_3).__name__}): Gewinner={r4_3.getWinner().name}")
print(f"Durch Los entschieden: {r4_3.wasDecidedByLot()}")
print(f"Runden gesamt: {election_3.getNumberOfIterations()}")

---
## Fall 4: Mehr Parteien als Wahlkreise

Nehmen mehr Parteien teil als Wahlkreise vorhanden sind, kann nicht jeder
Partei mit parlamentarischen Sitzen ein Wahlkreis zugewiesen werden. Die
Sitzverteilung erfolgt weiterhin proportional nach Stimmenanteil — die
Wahlkreiszuordnung jedoch höchstens für so viele Parteien, wie es Wahlkreise
gibt.

**Konfiguration:** 4 Parteien (A–D), 3 Wahlkreise à 100 Stimmen →
6 Parlamentssitze, Ballotmehrheit 52 %, parlamentarische Mehrheitsschwelle 55 %.

Stimmen (Wahlgang 1):

| Wahlkreis   | A                  | B                  | C                  | D                 |
|:-----------:|:------------------:|:------------------:|:------------------:|:-----------------:|
| WK1         | 56                 | 24                 | 17                 | 3                 |
| WK2         | 56                 | 24                 | 17                 | 3                 |
| WK3         | 55                 | 25                 | 16                 | 4                 |
| **Gesamt**  | **167 (55,7 %)** | **73 (24,3 %)** | **50 (16,7 %)** | **10 (3,3 %)** |

A gewinnt direkt im ersten Wahlgang (55,7 % ≥ 52 %). Da der Stimmenanteil die
parlamentarische Mehrheitsschwelle übertrifft (55,7 % > 55 %), werden alle Sitze
proportional verteilt.

Partei D erhält wegen zu geringem Stimmenanteil keinen Sitz. Partei C erhält
einen Sitz, aber — da es nur 3 Wahlkreise für 3 sitzberechtigte Parteien gibt —
keinen Wahlkreis: C ist im Parlament vertreten, ohne einen Direktmandatsträger zu stellen.

In [ ]:
from IPython.display import display

cc_f4 = ConstituenciesConfig.from_dataframe(pd.DataFrame({
    'constituency_name': ['WK1', 'WK2', 'WK3'],
    'constituency_size':  [100, 100, 100],
    'turnout_percent':    [100.0, 100.0, 100.0],
}))

config_f4 = ElectionConfig(
    constituencies_config=cc_f4,
    participating_parties=['A', 'B', 'C', 'D'],
    seed=42,
)
print(f"Wahlkreise: {cc_f4.getM()},  "
      f"Parteien: {len(config_f4.participating_parties)},  "
      f"Parlamentssitze: {config_f4.parliamentarySeats}")

election_f4 = Election(electionConfig=config_f4)
final_f4 = election_f4.start(votes={
    'WK1': {'A': 56, 'B': 24, 'C': 17, 'D': 3},
    'WK2': {'A': 56, 'B': 24, 'C': 17, 'D': 3},
    'WK3': {'A': 55, 'B': 25, 'C': 16, 'D': 4},
})
print(f"Gewinner: {final_f4.getWinner().name}  "
      f"(Runden: {election_f4.getNumberOfIterations()})")
display(final_f4.show_results_table(styler=True))

evaluator_f4 = ElectionEvaluator(seed=42)
result_f4 = evaluator_f4.evaluate(election_f4)

display(result_f4.get_seat_distribution_table())
display(result_f4.get_constituency_summary_table())
display(result_f4.get_constituency_assignment_table(sort_by='party'))